# Inverse PINN for the 1D Heat Equation — Estimating the Thermal Diffusivity $\alpha$

**Goal.** Recover the unknown thermal diffusivity $\alpha$ of a rod from a small number of noisy temperature
measurements, while forcing a neural network $u_\theta(x,t)$ to obey the heat equation. This is an *inverse problem*:
the physical law is known up to a parameter, and the data decide which parameter value is consistent with what we measured.

| | |
|---|---|
| PDE | $u_t = \alpha\, u_{xx}, \quad x\in[0,1],\ t\in[0,1]$ |
| Boundary conditions | $u(0,t) = u(1,t) = 0$ |
| Initial condition | $u(x,0) = \sin(\pi x)$ |
| Unknown | $\alpha>0$ &nbsp;(true value $\alpha_{\text{true}}=0.5$, hidden from training) |

**Structure**

* **Part 1 — Beginner foundation (sections 1–12).** Build every component step by step and train one baseline model.
* **Part 2 — Research mini-extension (sections 13–18).** Reproducible, isolated experiments on noise level, number of
  observations, loss weights and optimizers over several random seeds, followed by a results table, plots and a discussion.

**How to run.** `Runtime → Run all`. A GPU (`Runtime → Change runtime type → T4 GPU`) is recommended but not required.
After the baseline run, the notebook measures its own speed and prints an estimate of how long Part 2 will take;
set `QUICK_MODE = True` in section 14 for a smaller, faster sweep.

**Ground rule.** The exact solution $u(x,t)=e^{-\alpha\pi^2 t}\sin(\pi x)$ is used **only** to create the synthetic
measurements and to evaluate the result. It never enters a training loss — section 3 adds a guard that raises an
error if anyone tries.

---
# Part 1 — Beginner foundation

## 1. Imports, device, dtype and random seeds

* **Device.** We use the GPU if Colab provides one, otherwise the CPU. Every tensor is moved explicitly to `DEVICE`.
* **dtype.** All tensors use `DTYPE`, defined once here. `torch.float32` is fast; `torch.float64` is more precise
  (useful for L-BFGS) but slower on most GPUs.
* **Seeds.** Training involves randomness (initial weights, sampled points, noise). Fixing the seeds makes a run
  repeatable. On a GPU, tiny floating-point differences can remain between runs; they do not change the conclusions.

In [ ]:
# Standard library
import copy, csv, math, os, random, time
from contextlib import contextmanager
from dataclasses import dataclass, asdict, replace

# Pre-installed on Colab
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32          # change to torch.float64 for higher precision (slower on GPU)

def set_seed(seed: int) -> None:
    # Seed every random number generator we rely on.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)                     # seeds the CPU and all CUDA generators
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def sync() -> None:
    # GPU work is asynchronous; wait for it before reading a timer.
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def to_np(t: torch.Tensor) -> np.ndarray:
    return t.detach().cpu().numpy()

set_seed(0)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False, "font.size": 10})
print(f"PyTorch {torch.__version__} | device: {DEVICE} | dtype: {DTYPE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Configuration

Every setting lives in one **frozen** dataclass. *Frozen* means a configuration cannot be modified after it is created,
so no cell can change a setting behind another cell's back. To try a variant we create a new object, e.g.
`replace(cfg, noise_level=0.05)`. Part 2 relies on this: each experiment is fully described by its own `Config`.

In [ ]:
OBS_POOL_SIZE = 1000   # observations are drawn from a fixed pool so that smaller sets are subsets of larger ones

@dataclass(frozen=True)
class Config:
    # --- Physics ---------------------------------------------------------------------------
    alpha_true: float = 0.5      # ground truth: used ONLY to create synthetic data and to evaluate
    alpha_init: float = 0.1      # starting guess for the unknown alpha (deliberately wrong)
    # --- Data --------------------------------------------------------------------------------
    n_obs: int = 50              # number of noisy temperature measurements
    noise_level: float = 0.01    # 0.01 = 1 % noise (std = noise_level * std of the clean measurements)
    n_colloc: int = 2000         # interior points where the PDE residual is penalised
    n_bc: int = 200              # boundary points (half at x = 0, half at x = 1)
    n_ic: int = 200              # initial-condition points at t = 0
    # --- Network ---------------------------------------------------------------------------
    width: int = 50              # neurons per hidden layer
    depth: int = 4               # number of hidden layers
    # --- Loss weights -----------------------------------------------------------------------
    w_pde: float = 1.0
    w_bc: float = 1.0
    w_ic: float = 1.0
    w_data: float = 1.0
    # --- Optimisation -----------------------------------------------------------------------
    adam_iters: int = 5000
    lr_net: float = 1e-3         # Adam learning rate for the network weights
    lr_alpha: float = 5e-3       # Adam learning rate for log(alpha)
    lr_decay: float = 0.1        # both learning rates decay smoothly to lr_decay x their start value
    use_lbfgs: bool = False      # Part 2: optional L-BFGS refinement after Adam
    lbfgs_iters: int = 1000
    log_every: int = 50          # record losses and alpha every `log_every` iterations
    # --- Reproducibility and evaluation ------------------------------------------------------
    seed: int = 0
    n_eval: int = 101            # evaluation grid has n_eval x n_eval points

    def __post_init__(self):
        assert self.alpha_true > 0 and self.alpha_init > 0, "alpha must be positive"
        assert 0.0 <= self.noise_level < 1.0, "noise_level is a fraction, e.g. 0.05"
        assert 1 <= self.n_obs <= OBS_POOL_SIZE
        assert self.n_bc % 2 == 0, "n_bc must be even (half per boundary)"
        assert min(self.w_pde, self.w_bc, self.w_ic, self.w_data) >= 0, "loss weights must be >= 0"
        assert 0.0 < self.lr_decay <= 1.0, "lr_decay = final lr / initial lr, in (0, 1]"

cfg = Config()
for key, value in asdict(cfg).items():
    print(f"{key:>12s} = {value}")

## 3. The problem: PDE, boundary and initial conditions

The initial condition $\sin(\pi x)$ and the boundary value $0$ are **known physics**, so they may appear in the loss.
The exact solution

$$u(x,t) = e^{-\alpha_{\text{true}}\pi^2 t}\,\sin(\pi x)$$

is **not** allowed in the loss. `reference_solution` checks a guard flag: training code switches the guard on
(`with reference_locked(): ...`), and any call to the exact solution during training raises an error.
The cell ends with sanity checks that the exact solution satisfies the initial and boundary conditions, and that the guard works.

In [ ]:
PI = math.pi

def initial_condition(x: torch.Tensor) -> torch.Tensor:
    # u(x, 0) = sin(pi x). Known physics: allowed in the loss.
    return torch.sin(PI * x)

def boundary_value(t: torch.Tensor) -> torch.Tensor:
    # u(0, t) = u(1, t) = 0. Known physics: allowed in the loss.
    return torch.zeros_like(t)

class ReferenceGuard:
    locked = False           # True while a model is being trained

@contextmanager
def reference_locked():
    previous = ReferenceGuard.locked
    ReferenceGuard.locked = True
    try:
        yield
    finally:
        ReferenceGuard.locked = previous

def reference_solution(x: torch.Tensor, t: torch.Tensor, alpha: float) -> torch.Tensor:
    # Exact solution. ONLY for generating synthetic data and for evaluation/diagnostics.
    if ReferenceGuard.locked:
        raise RuntimeError("reference_solution() was called during training - the exact solution must not enter the loss.")
    return torch.exp(-alpha * PI**2 * t) * torch.sin(PI * x)

# Sanity checks --------------------------------------------------------------------------
_s = torch.linspace(0, 1, 11, dtype=DTYPE).reshape(-1, 1)
assert torch.allclose(reference_solution(_s, torch.zeros_like(_s), 0.5), initial_condition(_s), atol=1e-6)  # IC
assert torch.allclose(reference_solution(torch.zeros_like(_s), _s, 0.5), boundary_value(_s), atol=1e-6)   # x = 0
assert torch.allclose(reference_solution(torch.ones_like(_s), _s, 0.5), boundary_value(_s), atol=1e-6)    # x = 1
try:
    with reference_locked():
        reference_solution(_s, _s, 0.5)
    raise AssertionError("the guard did not fire")
except RuntimeError:
    pass
assert ReferenceGuard.locked is False
print("Problem definition OK: exact solution satisfies IC/BC, and the training guard works.")

## 4. Synthetic observations and training points

The PINN uses four sets of points:

| Set | Location | Loss term | Target |
|---|---|---|---|
| Collocation | random interior points | $\mathcal L_{\text{pde}}$ | PDE residual $=0$ |
| Boundary | $x=0$ and $x=1$, random $t$ | $\mathcal L_{\text{bc}}$ | $u=0$ |
| Initial | $t=0$, random $x$ | $\mathcal L_{\text{ic}}$ | $u=\sin(\pi x)$ |
| Observations | random $(x_i,t_i)$ | $\mathcal L_{\text{data}}$ | noisy measurement $\tilde u_i$ |

**Noise model.** $\tilde u_i = u(x_i,t_i) + \sigma\,\varepsilon_i$ with $\varepsilon_i\sim\mathcal N(0,1)$ and
$\sigma = \texttt{noise\_level}\times\operatorname{std}_i\,u(x_i,t_i)$ (the convention of Raissi et al., 2019).

**Reproducibility by design.** Each point set has its own random-number stream derived from the seed. For a fixed seed:
(i) the collocation, boundary and initial points do not change when `n_obs` or `noise_level` changes;
(ii) observation sets are *nested* — the 20 points are the first 20 of the 50, which are the first 50 of the 100;
(iii) all noise levels share the same locations and the same pattern $\varepsilon_i$, only $\sigma$ changes.
Comparisons in Part 2 are therefore *paired*. Data are generated on the CPU and then moved to `DEVICE`,
so a given seed produces identical data on CPU and GPU.

In [ ]:
STREAMS = {"obs_xt": 0, "obs_noise": 1, "colloc": 2, "bc": 3, "ic": 4}

def make_generator(seed: int, stream: str) -> torch.Generator:
    g = torch.Generator(device="cpu")
    g.manual_seed(1000 * seed + STREAMS[stream])
    return g

def rand_column(n: int, g: torch.Generator) -> torch.Tensor:
    return torch.rand(n, 1, generator=g, dtype=DTYPE)          # uniform on [0, 1), shape (n, 1)

def generate_observations(cfg: Config) -> dict:
    pool = torch.rand(OBS_POOL_SIZE, 2, generator=make_generator(cfg.seed, "obs_xt"), dtype=DTYPE)
    x = pool[:cfg.n_obs, 0:1].clone()
    t = pool[:cfg.n_obs, 1:2].clone()
    u_clean = reference_solution(x, t, cfg.alpha_true)          # allowed: this is data generation
    eps = torch.randn(OBS_POOL_SIZE, 1, generator=make_generator(cfg.seed, "obs_noise"), dtype=DTYPE)[:cfg.n_obs]
    noise_std = cfg.noise_level * u_clean.std().item()
    u_obs = u_clean + noise_std * eps
    # u_clean is kept for plots and diagnostics only; the loss uses u_obs.
    return {"x": x, "t": t, "u": u_obs, "u_clean": u_clean, "noise_std": noise_std}

def sample_collocation(cfg: Config) -> dict:
    g = make_generator(cfg.seed, "colloc")
    return {"x": rand_column(cfg.n_colloc, g), "t": rand_column(cfg.n_colloc, g)}

def sample_boundary(cfg: Config) -> dict:
    g = make_generator(cfg.seed, "bc")
    half = cfg.n_bc // 2
    t = rand_column(cfg.n_bc, g)
    x = torch.cat([torch.zeros(half, 1, dtype=DTYPE), torch.ones(half, 1, dtype=DTYPE)])
    return {"x": x, "t": t, "u": boundary_value(t)}

def sample_initial(cfg: Config) -> dict:
    g = make_generator(cfg.seed, "ic")
    x = rand_column(cfg.n_ic, g)
    return {"x": x, "t": torch.zeros_like(x), "u": initial_condition(x)}

def to_device(part: dict) -> dict:
    return {k: (v.to(device=DEVICE, dtype=DTYPE) if torch.is_tensor(v) else v) for k, v in part.items()}

def check_column(name: str, v: torch.Tensor, n: int) -> None:
    assert torch.is_tensor(v), f"{name} is not a tensor"
    assert tuple(v.shape) == (n, 1), f"{name}: expected shape ({n}, 1), got {tuple(v.shape)}"
    assert v.dtype == DTYPE, f"{name}: dtype {v.dtype} != {DTYPE}"
    assert v.device.type == DEVICE.type, f"{name}: on {v.device}, expected {DEVICE}"
    assert torch.isfinite(v).all(), f"{name} contains NaN/Inf"

def check_problem_data(data: dict, cfg: Config) -> None:
    obs, col, bc, ic = data["obs"], data["colloc"], data["bc"], data["ic"]
    for k in ("x", "t", "u", "u_clean"): check_column(f"obs.{k}", obs[k], cfg.n_obs)
    for k in ("x", "t"):                 check_column(f"colloc.{k}", col[k], cfg.n_colloc)
    for k in ("x", "t", "u"):            check_column(f"bc.{k}", bc[k], cfg.n_bc)
    for k in ("x", "t", "u"):            check_column(f"ic.{k}", ic[k], cfg.n_ic)
    for name, part in data.items():      # every point lies in the domain [0,1] x [0,1]
        for k in ("x", "t"):
            assert 0.0 <= part[k].min().item() and part[k].max().item() <= 1.0, f"{name}.{k} outside [0, 1]"
    half = cfg.n_bc // 2                 # boundary points: exactly x = 0 (first half), x = 1 (second half), target 0
    assert torch.all(bc["x"][:half] == 0.0) and torch.all(bc["x"][half:] == 1.0), "boundary x must be exactly 0 or 1"
    assert torch.all(bc["u"] == 0.0), "boundary target must be 0"
    assert torch.all(ic["t"] == 0.0), "initial points must have t = 0 exactly"       # initial points
    assert torch.allclose(ic["u"], initial_condition(ic["x"])), "initial target must be sin(pi x)"
    if cfg.noise_level == 0.0:
        assert torch.equal(obs["u"], obs["u_clean"]), "0 % noise must give clean data"
    else:
        assert not torch.equal(obs["u"], obs["u_clean"]), "noise was not added"

def build_problem_data(cfg: Config) -> dict:
    data = {"obs": generate_observations(cfg), "colloc": sample_collocation(cfg),
            "bc": sample_boundary(cfg), "ic": sample_initial(cfg)}
    data = {name: to_device(part) for name, part in data.items()}
    check_problem_data(data, cfg)
    return data

data = build_problem_data(cfg)
print(f"observations : {cfg.n_obs}  (noise level {cfg.noise_level:.0%}, noise std = {data['obs']['noise_std']:.4f})")
print(f"collocation  : {cfg.n_colloc}\nboundary     : {cfg.n_bc}\ninitial      : {cfg.n_ic}")

# Demonstrate the nesting property promised above
_small = generate_observations(replace(cfg, n_obs=20))
assert torch.equal(_small["x"], data["obs"]["x"][:20].cpu()) and torch.equal(_small["t"], data["obs"]["t"][:20].cpu())
print("All data sanity checks passed (shapes, dtype, device, domain, boundary/initial construction, nesting).")

## 5. Look at the data

Left: the reference field (for our eyes only) with the measurement locations. Middle: where each loss term is
evaluated. Right: the noisy measurements against exact temperature profiles at a few times. Notice how fast the
temperature decays: by $t=1$ the amplitude is $e^{-0.5\pi^2}\approx 0.007$, only about four times the 1 % noise level and below the 5 % noise level,
so late-time measurements carry little information about $\alpha$ (we return to this in section 17).

In [ ]:
def eval_grid(n: int):
    # Space-time grid; rows are times, columns are positions. Returned on the CPU.
    xs = torch.linspace(0, 1, n, dtype=DTYPE)
    ts = torch.linspace(0, 1, n, dtype=DTYPE)
    T, X = torch.meshgrid(ts, xs, indexing="ij")
    return X, T

def plot_problem_setup(data: dict, cfg: Config) -> None:
    X, T = eval_grid(cfg.n_eval)
    U = reference_solution(X, T, cfg.alpha_true)
    obs, col, bc, ic = data["obs"], data["colloc"], data["bc"], data["ic"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

    ax = axes[0]
    im = ax.pcolormesh(to_np(X), to_np(T), to_np(U), shading="auto", cmap="inferno")
    ax.scatter(to_np(obs["x"]), to_np(obs["t"]), s=22, c="cyan", edgecolors="k", linewidths=0.5, label="measurements")
    fig.colorbar(im, ax=ax, label="u")
    ax.set(xlabel="x", ylabel="t", title="Reference field (never used in training)")
    ax.legend(loc="upper right", fontsize=8)

    ax = axes[1]
    ax.scatter(to_np(col["x"]), to_np(col["t"]), s=2, c="0.75", label=f"collocation ({cfg.n_colloc})")
    ax.scatter(to_np(bc["x"]), to_np(bc["t"]), s=8, c="tab:blue", label=f"boundary ({cfg.n_bc})")
    ax.scatter(to_np(ic["x"]), to_np(ic["t"]), s=8, c="tab:green", label=f"initial ({cfg.n_ic})")
    ax.scatter(to_np(obs["x"]), to_np(obs["t"]), s=30, c="tab:red", marker="x", label=f"measurements ({cfg.n_obs})")
    ax.set(xlabel="x", ylabel="t", xlim=(-0.04, 1.04), ylim=(-0.04, 1.04), title="Where each loss term is evaluated")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, fontsize=8)

    ax = axes[2]
    xs = torch.linspace(0, 1, 200, dtype=DTYPE).reshape(-1, 1)
    cmap = plt.get_cmap("viridis")
    for tk in (0.0, 0.1, 0.25, 0.5, 1.0):
        ax.plot(to_np(xs), to_np(reference_solution(xs, torch.full_like(xs, tk), cfg.alpha_true)),
                color=cmap(tk), lw=1.5, label=f"exact, t = {tk}")
    sc = ax.scatter(to_np(obs["x"]), to_np(obs["u"]), c=to_np(obs["t"]), cmap="viridis", vmin=0, vmax=1,
                    s=25, edgecolors="k", linewidths=0.5)
    fig.colorbar(sc, ax=ax, label="time of measurement")
    ax.set(xlabel="x", ylabel="u", title=f"Noisy measurements ({cfg.noise_level:.0%} noise)")
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

plot_problem_setup(data, cfg)

## 6. The neural network $u_\theta(x,t)$ and the unknown $\alpha$

* **Architecture.** A fully connected network maps $(x,t)\mapsto u$: 4 hidden layers of 50 neurons.
* **Why `tanh`?** The loss needs the second derivative $u_{xx}$. With ReLU, $u_{xx}=0$ almost everywhere, so the
  physics loss would be meaningless. `tanh` is smooth, so all derivatives exist.
* **Input scaling.** Inputs are mapped from $[0,1]$ to $[-1,1]$, where `tanh` is most sensitive.
* **Initialisation.** Xavier (Glorot) initialisation keeps signals at a reasonable size through the layers.
* **Positive $\alpha$.** We do not train $\alpha$ directly. We train $\log\alpha$ and use $\alpha=\exp(\log\alpha)$, which is
  positive for every value of the trainable parameter. Updates to $\log\alpha$ are *relative* changes of $\alpha$,
  which suits a parameter whose scale we do not know in advance.

In [ ]:
class MLP(nn.Module):
    # Fully connected tanh network: (x, t) -> u.
    def __init__(self, width: int, depth: int):
        super().__init__()
        dims = [2] + [width] * depth + [1]
        layers = []
        for i, (d_in, d_out) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(nn.Linear(d_in, d_out))
            if i < len(dims) - 2:                  # no activation after the output layer
                layers.append(nn.Tanh())
        self.layers = nn.Sequential(*layers)
        for m in self.layers:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        z = torch.cat([x, t], dim=1)               # (N, 2)
        z = 2.0 * z - 1.0                          # [0, 1] -> [-1, 1]
        return self.layers(z)                      # (N, 1)

class InversePINN(nn.Module):
    # Network u_theta(x, t) plus the trainable physical parameter log(alpha).
    def __init__(self, cfg: Config):
        super().__init__()
        self.u_net = MLP(cfg.width, cfg.depth)
        self.log_alpha = nn.Parameter(torch.tensor(math.log(cfg.alpha_init), dtype=DTYPE))

    @property
    def alpha(self) -> torch.Tensor:
        return torch.exp(self.log_alpha)           # always > 0

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        return self.u_net(x, t)

def build_model(cfg: Config) -> InversePINN:
    torch.manual_seed(cfg.seed)                    # initial weights depend only on the seed
    return InversePINN(cfg).to(device=DEVICE, dtype=DTYPE)

# Sanity checks ------------------------------------------------------------------------
model = build_model(cfg)
u_test = model(data["colloc"]["x"][:5], data["colloc"]["t"][:5])
assert u_test.shape == (5, 1) and u_test.dtype == DTYPE and u_test.device.type == DEVICE.type
assert model.alpha.item() > 0 and abs(model.alpha.item() - cfg.alpha_init) < 1e-6
print(model)
print(f"trainable parameters: {sum(p.numel() for p in model.parameters()):,} (network weights + log_alpha)")
print(f"initial alpha = {model.alpha.item():.3f}   (true value {cfg.alpha_true}, hidden from training)")

## 7. Derivatives with automatic differentiation

`torch.autograd.grad(u, x, grad_outputs=ones)` returns $\partial u/\partial x$ at every point in the batch. This works
because $u_i$ depends only on its own input $(x_i,t_i)$, so the gradient of $\sum_j u_j$ with respect to $x_i$ equals
$\partial u_i/\partial x_i$.

`create_graph=True` is needed for two reasons: (1) to differentiate $u_x$ once more and obtain $u_{xx}$;
(2) so that the loss, which contains $u_t$ and $u_{xx}$, can itself be back-propagated to $\theta$ and $\alpha$.

**Sanity check.** Before trusting the code on a network, we test it on a function whose derivatives we know:
for the exact solution, $u_t=-\alpha\pi^2u$ and $u_{xx}=-\pi^2u$, so the residual must vanish for the correct
$\alpha$ and not for a wrong one. (Using the exact solution to *test code* is fine; it is never used in training.)

In [ ]:
def gradient(y: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    # dy/dx for a batch; the graph is kept so we can differentiate again and backpropagate.
    return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True)[0]

def pde_derivatives(u_fn, x: torch.Tensor, t: torch.Tensor):
    # Return u, u_t, u_xx at the points (x, t); u_fn can be the network or any function of (x, t).
    x = x.detach().clone().requires_grad_(True)
    t = t.detach().clone().requires_grad_(True)
    u = u_fn(x, t)
    u_t = gradient(u, t)
    u_x = gradient(u, x)
    u_xx = gradient(u_x, x)
    return u, u_t, u_xx

def pde_residual(u_fn, alpha, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    # r = u_t - alpha * u_xx  (zero when u solves the heat equation with this alpha)
    _, u_t, u_xx = pde_derivatives(u_fn, x, t)
    return u_t - alpha * u_xx

# Sanity check 1: derivatives of a function with known derivatives --------------------
g_chk = torch.Generator(device="cpu"); g_chk.manual_seed(123)
x_chk = torch.rand(256, 1, generator=g_chk, dtype=DTYPE).to(DEVICE)
t_chk = torch.rand(256, 1, generator=g_chk, dtype=DTYPE).to(DEVICE)
exact = lambda x, t: reference_solution(x, t, 0.5)
u, u_t, u_xx = pde_derivatives(exact, x_chk, t_chk)
assert u.shape == u_t.shape == u_xx.shape == (256, 1)
assert torch.allclose(u_t, -0.5 * PI**2 * u, atol=1e-4), "u_t is wrong"
assert torch.allclose(u_xx, -PI**2 * u, atol=1e-4), "u_xx is wrong"
r_right = pde_residual(exact, 0.5, x_chk, t_chk).abs().max().item()
r_wrong = pde_residual(exact, 0.3, x_chk, t_chk).abs().max().item()
assert r_right < 1e-4 < r_wrong
print(f"max |residual| of exact solution: correct alpha {r_right:.1e}, wrong alpha {r_wrong:.1e}")

# Sanity check 2: gradients flow from the residual back to the network AND to log_alpha
r_net = pde_residual(model, model.alpha, x_chk, t_chk)
assert r_net.shape == (256, 1)
(r_net ** 2).mean().backward()
assert model.log_alpha.grad is not None and torch.isfinite(model.log_alpha.grad)
assert all(m.weight.grad is not None for m in model.u_net.layers if isinstance(m, nn.Linear))
# (The output-layer bias gets no gradient here: adding a constant to u does not change u_t or u_xx.
#  It is trained by the boundary, initial and data terms.)
model.zero_grad(set_to_none=True)
print("Derivative checks passed.")

## 8. Loss functions

$$
\mathcal L(\theta,\alpha)=w_{\text{pde}}\,\mathcal L_{\text{pde}}+w_{\text{bc}}\,\mathcal L_{\text{bc}}
+w_{\text{ic}}\,\mathcal L_{\text{ic}}+w_{\text{data}}\,\mathcal L_{\text{data}}
$$

$$
\mathcal L_{\text{pde}}=\frac1{N_f}\sum_{j}\big(\partial_t u_\theta-\alpha\,\partial_{xx}u_\theta\big)^2(x_j,t_j),\qquad
\mathcal L_{\text{bc}}=\frac1{N_b}\sum_{j}u_\theta(x^b_j,t^b_j)^2,
$$
$$
\mathcal L_{\text{ic}}=\frac1{N_0}\sum_{j}\big(u_\theta(x^0_j,0)-\sin(\pi x^0_j)\big)^2,\qquad
\mathcal L_{\text{data}}=\frac1{N_d}\sum_{i}\big(u_\theta(x_i,t_i)-\tilde u_i\big)^2 .
$$

The components are returned **separately** (unweighted), so we can monitor physics, boundary, initial and data
fit individually. Note one structural fact that matters later: **$\alpha$ appears only in $\mathcal L_{\text{pde}}$.**
The data never touch $\alpha$ directly; they shape $u_\theta$, and the physics term then adjusts $\alpha$ until it
matches the decay rate the network has learned from the data.

In [ ]:
LOSS_NAMES = ("pde", "bc", "ic", "data")

def loss_weights(cfg: Config) -> dict:
    return {"pde": cfg.w_pde, "bc": cfg.w_bc, "ic": cfg.w_ic, "data": cfg.w_data}

def compute_losses(model: InversePINN, data: dict, cfg: Config):
    # Return (weighted total loss, dict of unweighted components).
    col, bc, ic, obs = data["colloc"], data["bc"], data["ic"], data["obs"]
    r = pde_residual(model, model.alpha, col["x"], col["t"])
    losses = {
        "pde":  torch.mean(r ** 2),
        "bc":   torch.mean((model(bc["x"], bc["t"]) - bc["u"]) ** 2),
        "ic":   torch.mean((model(ic["x"], ic["t"]) - ic["u"]) ** 2),
        "data": torch.mean((model(obs["x"], obs["t"]) - obs["u"]) ** 2),   # noisy measurements only
    }
    w = loss_weights(cfg)
    total = sum(w[k] * losses[k] for k in LOSS_NAMES)
    return total, losses

# Sanity check on the untrained model (inside the guard, exactly as during training)
with reference_locked():
    total, parts = compute_losses(model, data, cfg)
assert total.ndim == 0 and torch.isfinite(total)
print("untrained model:", "  ".join(f"L_{k} = {v.item():.3e}" for k, v in parts.items()), f"  total = {total.item():.3e}")

## 9. Training

**Adam** is used with two parameter groups: the network weights (`lr_net`) and $\log\alpha$ (`lr_alpha`).
$\log\alpha$ is a single number that must travel from $\log 0.1$ to about $\log 0.5$, so it gets a somewhat larger step size.
Both learning rates **decay exponentially** to 10 % of their initial value by the last iteration. With a constant
learning rate, Adam keeps taking steps of the same size near the optimum and the loss shows occasional spikes late in
training; the decay lets the final iterations settle.
Every `log_every` iterations we record the total loss, each (unweighted) component, and the current $\alpha$.
Training runs inside `reference_locked()`, and we check that the loss and $\alpha$ stay finite.

**L-BFGS** (used in Part 2) is a quasi-Newton method: it builds a curvature estimate from recent gradients and
often reaches a much lower loss than Adam in a few hundred steps, which is why "Adam, then L-BFGS" is a common PINN
recipe. It can also fail (NaN, or line-search trouble). We therefore save a snapshot before L-BFGS and restore it if the
loss becomes non-finite or ends higher than it started. In the logs, the L-BFGS x-axis counts *function evaluations*.

In [ ]:
def new_history() -> dict:
    return {"iter": [], "phase": [], "total": [], "alpha": [], **{k: [] for k in LOSS_NAMES}}

def log_step(history: dict, it: int, phase: str, total, parts: dict, alpha) -> None:
    history["iter"].append(it)
    history["phase"].append(phase)
    history["total"].append(total.item())
    history["alpha"].append(alpha.item())
    for k in LOSS_NAMES:
        history[k].append(parts[k].item())

def assert_finite(model: InversePINN, total: torch.Tensor) -> None:
    assert torch.isfinite(total), "loss became NaN/Inf - try a smaller learning rate"
    assert torch.isfinite(model.log_alpha) and model.alpha.item() > 0, "alpha is no longer a positive finite number"

def train_adam(model: InversePINN, data: dict, cfg: Config, history: dict, verbose: bool = True) -> None:
    optimizer = torch.optim.Adam([
        {"params": model.u_net.parameters(), "lr": cfg.lr_net},
        {"params": [model.log_alpha],        "lr": cfg.lr_alpha},
    ])
    # Exponential decay: after adam_iters steps each learning rate equals lr_decay x its initial value
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=cfg.lr_decay ** (1.0 / cfg.adam_iters))
    with reference_locked():
        for it in range(1, cfg.adam_iters + 1):
            optimizer.zero_grad(set_to_none=True)
            total, parts = compute_losses(model, data, cfg)
            total.backward()
            if it == 1 or it % cfg.log_every == 0 or it == cfg.adam_iters:
                log_step(history, it, "adam", total, parts, model.alpha)   # values before this update
                assert_finite(model, total)
                if verbose and (it == 1 or it % (20 * cfg.log_every) == 0 or it == cfg.adam_iters):
                    print(f"[Adam {it:5d}] total {total.item():.3e} | pde {parts['pde'].item():.2e} "
                          f"bc {parts['bc'].item():.2e} ic {parts['ic'].item():.2e} "
                          f"data {parts['data'].item():.2e} | alpha {model.alpha.item():.5f}")
            optimizer.step()
            scheduler.step()

def train_lbfgs(model: InversePINN, data: dict, cfg: Config, history: dict, verbose: bool = True):
    # L-BFGS refinement; returns (status, number of function evaluations). Status: ok / reverted / failed.
    snapshot = copy.deepcopy(model.state_dict())
    start = history["iter"][-1] if history["iter"] else 0
    optimizer = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=cfg.lbfgs_iters,
                                  max_eval=int(1.25 * cfg.lbfgs_iters), history_size=50,
                                  tolerance_grad=1e-9, tolerance_change=1e-12, line_search_fn="strong_wolfe")
    n_evals = 0
    def closure():
        nonlocal n_evals
        optimizer.zero_grad(set_to_none=True)
        total, parts = compute_losses(model, data, cfg)
        total.backward()
        n_evals += 1
        if n_evals % cfg.log_every == 0:
            log_step(history, start + n_evals, "lbfgs", total, parts, model.alpha)
        return total

    status = "ok"
    with reference_locked():
        loss_before = compute_losses(model, data, cfg)[0].item()
        try:
            optimizer.step(closure)
        except RuntimeError as err:
            status = f"failed ({type(err).__name__})"
        if status == "ok":
            total, parts = compute_losses(model, data, cfg)
            loss_after = total.item()
            if not (math.isfinite(loss_after) and loss_after <= loss_before and torch.isfinite(model.log_alpha)):
                status = "reverted"
            else:
                log_step(history, start + n_evals, "lbfgs", total, parts, model.alpha)
    if status != "ok":
        model.load_state_dict(snapshot)            # fall back to the Adam solution
    if verbose:
        print(f"[L-BFGS] status: {status} | {n_evals} function evaluations | alpha {model.alpha.item():.5f}")
    return status, n_evals

## 10. Evaluation metrics

On a uniform $101\times101$ grid over $[0,1]^2$ (the exact solution is allowed here, training is over):

* **$\alpha$ relative error** $\;|\hat\alpha-\alpha_{\text{true}}|/\alpha_{\text{true}}$ — the main quantity of interest.
* **Field MSE** and **relative $L^2$ error** $\;\|u_\theta-u\|_2/\|u\|_2$ — how well the whole temperature field is reconstructed.
* **Residual MSE** — mean of $(\partial_t u_\theta-\hat\alpha\,\partial_{xx}u_\theta)^2$ with the *learned* $\hat\alpha$:
  how well the network satisfies the PDE it believes in. Measured on the grid, i.e. mostly away from the training collocation points.

In [ ]:
def evaluate(model: InversePINN, cfg: Config, chunk: int = 4096):
    X, T = eval_grid(cfg.n_eval)
    x = X.reshape(-1, 1).to(DEVICE)
    t = T.reshape(-1, 1).to(DEVICE)
    with torch.no_grad():
        u_pred = model(x, t)
    u_ref = reference_solution(x, t, cfg.alpha_true)
    residual = torch.cat([pde_residual(model, model.alpha, x[i:i + chunk], t[i:i + chunk]).detach()
                          for i in range(0, x.shape[0], chunk)])
    assert u_pred.shape == u_ref.shape == residual.shape == (cfg.n_eval ** 2, 1)
    err = u_pred - u_ref
    alpha_hat = model.alpha.item()
    metrics = {
        "alpha": alpha_hat,
        "alpha_rel_err": abs(alpha_hat - cfg.alpha_true) / cfg.alpha_true,
        "field_mse": torch.mean(err ** 2).item(),
        "field_rel_l2": (torch.linalg.norm(err) / torch.linalg.norm(u_ref)).item(),
        "residual_mse": torch.mean(residual ** 2).item(),
    }
    shape = tuple(X.shape)
    fields = {"X": to_np(X), "T": to_np(T), "u_pred": to_np(u_pred).reshape(shape),
              "u_ref": to_np(u_ref).reshape(shape), "abs_err": to_np(err.abs()).reshape(shape),
              "residual": to_np(residual).reshape(shape)}
    return metrics, fields

## 11. Plotting helpers

Two figures: (a) training curves — each loss component on a log scale, and the trajectory of $\alpha$;
(b) fields — prediction, reference, pointwise error, PDE residual, temperature profiles, and the decay of $u(0.5,t)$.
On a log scale that decay is a straight line with slope $-\alpha\pi^2$, which is exactly the information $\alpha$ is inferred from.

In [ ]:
LOSS_STYLE = {"pde": ("tab:red", r"$\mathcal{L}_{pde}$"), "bc": ("tab:blue", r"$\mathcal{L}_{bc}$"),
              "ic": ("tab:green", r"$\mathcal{L}_{ic}$"), "data": ("tab:purple", r"$\mathcal{L}_{data}$")}

def plot_training_history(history: dict, cfg: Config, title: str = "") -> None:
    it = np.asarray(history["iter"])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    ax = axes[0]
    ax.semilogy(it, history["total"], color="k", lw=2, label="total (weighted)")
    for k in LOSS_NAMES:
        color, label = LOSS_STYLE[k]
        ax.semilogy(it, history[k], color=color, lw=1, label=label + " (unweighted)")
    ax2 = axes[1]
    ax2.plot(it, history["alpha"], color="tab:orange", lw=2, label=r"learned $\alpha$")
    ax2.axhline(cfg.alpha_true, color="k", ls="--", lw=1, label=r"$\alpha_{true}$")
    if "lbfgs" in history["phase"]:
        start = it[history["phase"].index("lbfgs")]
        for a in axes:
            a.axvline(start, color="0.5", ls=":", lw=1)
            a.text(start, 0.97, " L-BFGS", transform=a.get_xaxis_transform(), va="top", fontsize=8, color="0.4")
    ax.set(xlabel="iteration", ylabel="loss", title="Loss components")
    ax.legend(fontsize=8, ncol=2)
    ax2.set(xlabel="iteration", ylabel=r"$\alpha$", title=r"Convergence of $\alpha$")
    ax2.legend(fontsize=8)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()

def plot_fields(fields: dict, metrics: dict, cfg: Config, data: dict = None) -> None:
    X, T = fields["X"], fields["T"]
    vmin, vmax = fields["u_ref"].min(), fields["u_ref"].max()
    rmax = np.abs(fields["residual"]).max()
    fig, axes = plt.subplots(2, 3, figsize=(16, 8.6))
    panels = [(axes[0, 0], "u_pred", rf"PINN prediction ($\hat\alpha$ = {metrics['alpha']:.4f})", "inferno", vmin, vmax),
              (axes[0, 1], "u_ref", r"Reference ($\alpha_{true}$" + f" = {cfg.alpha_true})", "inferno", vmin, vmax),
              (axes[0, 2], "abs_err", f"|prediction - reference|  (rel. L2 = {metrics['field_rel_l2']:.2e})", "viridis", None, None),
              (axes[1, 0], "residual", f"PDE residual  (MSE = {metrics['residual_mse']:.2e})", "RdBu_r", -rmax, rmax)]
    for ax, key, title, cmap, lo, hi in panels:
        im = ax.pcolormesh(X, T, fields[key], shading="auto", cmap=cmap, vmin=lo, vmax=hi)
        fig.colorbar(im, ax=ax)
        ax.set(xlabel="x", ylabel="t", title=title)

    ax = axes[1, 1]                      # profiles u(x, t_k)
    cmap = plt.get_cmap("viridis")
    xs = X[0]
    for tk in (0.0, 0.1, 0.25, 0.5):
        row = int(round(tk * (cfg.n_eval - 1)))
        ax.plot(xs, fields["u_ref"][row], color=cmap(tk), lw=2.5, alpha=0.45)
        ax.plot(xs, fields["u_pred"][row], color=cmap(tk), lw=1.2, ls="--", label=f"t = {tk}")
    ax.set(xlabel="x", ylabel="u", title="Profiles: reference (thick) vs PINN (dashed)")
    ax.legend(fontsize=8)

    ax = axes[1, 2]                      # decay at x = 0.5
    col = (cfg.n_eval - 1) // 2
    ts = T[:, col]
    ax.semilogy(ts, fields["u_ref"][:, col], color="k", lw=2.5, alpha=0.45, label="reference")
    ax.semilogy(ts, np.clip(fields["u_pred"][:, col], 1e-6, None), color="tab:orange", ls="--", label="PINN")
    if data is not None:
        obs = data["obs"]
        near = (obs["x"] - 0.5).abs().squeeze(1) < 0.1
        if near.any():
            ax.semilogy(to_np(obs["t"][near]).ravel(), np.clip(to_np(obs["u"][near]).ravel(), 1e-6, None), "o",
                        ms=4, color="tab:purple", label="measurements, |x-0.5|<0.1")
    ax.set(xlabel="t", ylabel="u(0.5, t)", title=r"Decay at $x=0.5$: slope $=-\alpha\pi^2$")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 12. Baseline experiment

Baseline: $\alpha_{\text{init}}=0.1$, 50 measurements with 1 % noise, all loss weights equal to 1, 5000 Adam iterations.
Everything below uses the functions defined above — nothing else is hidden.

In [ ]:
cfg = Config()                      # baseline settings
set_seed(cfg.seed)
data = build_problem_data(cfg)
model = build_model(cfg)
history = new_history()

sync(); t0 = time.perf_counter()
train_adam(model, data, cfg, history)
sync(); baseline_runtime = time.perf_counter() - t0

metrics, fields = evaluate(model, cfg)
with reference_locked():
    _, final_parts = compute_losses(model, data, cfg)

print(f"\nTraining time: {baseline_runtime:.1f} s on {DEVICE}")
print(f"learned alpha      : {metrics['alpha']:.5f}   (true {cfg.alpha_true})")
print(f"alpha rel. error   : {metrics['alpha_rel_err']:.3e}")
print(f"field MSE          : {metrics['field_mse']:.3e}")
print(f"field rel. L2 error: {metrics['field_rel_l2']:.3e}")
print(f"residual MSE (grid): {metrics['residual_mse']:.3e}")
print("\nFinal unweighted loss components (reported separately):")
for k in LOSS_NAMES:
    print(f"  L_{k:<5s}= {final_parts[k].item():.3e}")
print(f"\nData-fit RMSE {math.sqrt(final_parts['data'].item()):.4f} vs noise std {data['obs']['noise_std']:.4f} "
      "(an RMSE far below the noise std would indicate over-fitting the noise).")
assert metrics["alpha"] > 0

In [ ]:
plot_training_history(history, cfg, title="Baseline: 50 measurements, 1 % noise, Adam")
plot_fields(fields, metrics, cfg, data)

**Reading the baseline.**

* $\alpha$ moves away from the initial guess 0.1 quickly and settles near 0.5. The printed relative error is the
  number to quote; it will differ slightly between seeds and hardware (Part 2 quantifies this).
* The four loss components are reported separately. $\mathcal L_{\text{data}}$ levels off near the noise variance
  $\sigma^2$: the network cannot, and should not, fit the noise exactly.
* The pointwise error is largest where the solution changes fastest (small $t$, middle of the rod).
* **Things to try:** start from `alpha_init=2.0`; set `noise_level=0.0`; set `w_pde=0` (the network then fits the
  data but $\alpha$ receives no gradient at all — why?).

---
# Part 2 — Research mini-extension

## 13. An isolated, reproducible experiment runner

Design principles:

* **Isolation.** `run_experiment(cfg)` does everything from scratch — seeding, data, model, optimizer, training,
  evaluation — and depends only on its `Config`. Nothing leaks in from Part 1 or from earlier runs; the check at the
  end of the cell trains the same configuration twice, with the global random state disturbed in between, and
  verifies that the results agree.
* **One factor at a time.** Each experiment changes a single factor relative to the baseline
  (1 % noise, 50 measurements, weights $(1,1,1,1)$, Adam only). This keeps the number of runs small and the effects
  interpretable; interactions between factors are not studied.
* **Several seeds.** A seed changes the initial weights and every sampled point set. We report mean $\pm$ sample
  standard deviation over seeds.
* **Paired data and caching.** For a given seed, settings share data wherever possible (section 4). Identical
  configurations — the baseline appears in all four experiments — are trained once and reused.

**A statistical yardstick: the Cramér–Rao lower bound (CRLB).** Imagine an idealised estimator that already knows the
solution family $u(x,t;\alpha)=e^{-\alpha\pi^2t}\sin(\pi x)$ and only has to fit $\alpha$ to measurements with Gaussian
noise of standard deviation $\sigma$. No unbiased estimator can then do better than

$$
\operatorname{sd}(\hat\alpha)\;\ge\;\frac{\sigma}{\sqrt{\sum_i s_i^2}},\qquad
s_i=\frac{\partial u}{\partial\alpha}(x_i,t_i)=-\pi^2\,t_i\,u(x_i,t_i).
$$

The PINN has a harder job — it must also learn the whole field — so its error should lie *above* this bound. The gap
measures the cost of learning the field plus optimisation error. (This is analysis only; it uses the exact solution
and plays no role in training.)

In [ ]:
def alpha_crlb_rel(data: dict, cfg: Config) -> float:
    # CRLB on sd(alpha_hat) / alpha_true for the idealised estimator described above. Analysis only.
    if cfg.noise_level == 0.0:
        return 0.0
    x, t = data["obs"]["x"], data["obs"]["t"]
    s = -PI**2 * t * reference_solution(x, t, cfg.alpha_true)          # du/dalpha at the measurement points
    sd = data["obs"]["noise_std"] / torch.sqrt(torch.sum(s ** 2)).item()
    return sd / cfg.alpha_true

METRIC_KEYS = ("alpha", "alpha_rel_err", "field_mse", "field_rel_l2", "residual_mse", "runtime_s", "alpha_crlb_rel")
_RUN_CACHE = {}          # Config -> result; Config is frozen, hence hashable

def run_experiment(cfg: Config, use_cache: bool = True) -> dict:
    # Train and evaluate one configuration from scratch. The result depends only on cfg.
    if use_cache and cfg in _RUN_CACHE:
        return _RUN_CACHE[cfg]
    set_seed(cfg.seed)
    data = build_problem_data(cfg)
    model = build_model(cfg)
    history = new_history()
    sync(); t0 = time.perf_counter()
    train_adam(model, data, cfg, history, verbose=False)
    lbfgs_status, lbfgs_evals = "not used", 0
    if cfg.use_lbfgs:
        lbfgs_status, lbfgs_evals = train_lbfgs(model, data, cfg, history, verbose=False)
    sync(); runtime = time.perf_counter() - t0
    metrics, _ = evaluate(model, cfg)
    with reference_locked():
        _, parts = compute_losses(model, data, cfg)
    result = {**metrics, "runtime_s": runtime, "alpha_crlb_rel": alpha_crlb_rel(data, cfg),
              "lbfgs_status": lbfgs_status, "lbfgs_evals": lbfgs_evals,
              "final_losses": {k: v.item() for k, v in parts.items()}, "history": history, "cfg": cfg}
    assert result["alpha"] > 0 and all(math.isfinite(result[k]) for k in METRIC_KEYS)
    _RUN_CACHE[cfg] = result
    return result

# Isolation check: same config, trained twice from scratch, with the global RNG disturbed in between
_short = replace(Config(), adam_iters=100)
_a = run_experiment(_short, use_cache=False)["alpha"]
torch.rand(1000); np.random.rand(1000); random.random()
_b = run_experiment(_short, use_cache=False)["alpha"]
assert abs(_a - _b) < 1e-4, f"runs are not reproducible: {_a} vs {_b}"
print(f"Isolation check passed: alpha after 100 iterations = {_a:.6f} and {_b:.6f}")

## 14. Experiment design and execution

| Experiment | Values | Everything else |
|---|---|---|
| Noise level | 0 %, 1 %, 5 % | baseline |
| Number of observations | 20, 50, 100 | baseline |
| Loss weights $(w_{\text{pde}},w_{\text{bc}},w_{\text{ic}},w_{\text{data}})$ | $(1,1,1,1)$, $(1,1,1,10)$, $(10,1,1,1)$, $(0.1,1,1,1)$ | baseline |
| Optimizer | Adam (5000 it.) vs Adam (5000 it.) + L-BFGS (≤1000 it.) | baseline |

Seeds: 0, 1, 2. The cell first prints how many training runs are needed and an estimate of the time, based on the
speed measured in section 12. **On a CPU runtime, consider `QUICK_MODE = True`** (2000 Adam iterations, 2 seeds);
results are then noisier and less converged.

In [ ]:
QUICK_MODE = False

SEEDS = (0, 1) if QUICK_MODE else (0, 1, 2)
BASE = Config(adam_iters=2000 if QUICK_MODE else 5000, lbfgs_iters=500 if QUICK_MODE else 1000)

EXPERIMENTS = {
    "noise": [(f"{p:.0%} noise", replace(BASE, noise_level=p)) for p in (0.0, 0.01, 0.05)],
    "n_obs": [(f"N_obs = {n}", replace(BASE, n_obs=n)) for n in (20, 50, 100)],
    "weights": [(label, replace(BASE, **w)) for label, w in (
        ("balanced (1,1,1,1)", {}),
        ("data-heavy (1,1,1,10)", {"w_data": 10.0}),
        ("physics-heavy (10,1,1,1)", {"w_pde": 10.0}),
        ("physics-light (0.1,1,1,1)", {"w_pde": 0.1}),
    )],
    "optimizer": [("Adam", BASE), ("Adam + L-BFGS", replace(BASE, use_lbfgs=True))],
}
EXP_TITLES = {"noise": "Noise level", "n_obs": "Number of observations",
              "weights": "Loss weights (pde, bc, ic, data)", "optimizer": "Optimizer"}

unique_cfgs = {replace(c, seed=s) for settings in EXPERIMENTS.values() for _, c in settings for s in SEEDS}
todo = [c for c in unique_cfgs if c not in _RUN_CACHE]
sec_per_step = baseline_runtime / cfg.adam_iters                     # measured in section 12
estimate = sec_per_step * sum(c.adam_iters + (1.25 * c.lbfgs_iters if c.use_lbfgs else 0) for c in todo)
print(f"{len(unique_cfgs)} unique training runs ({len(todo)} still to train); "
      f"rough time estimate on this {DEVICE.type.upper()}: {estimate / 60:.0f} min\n")

all_runs = []
t_start = time.perf_counter()
for exp_name, settings in EXPERIMENTS.items():
    for label, setting_cfg in settings:
        for seed in SEEDS:
            run_cfg = replace(setting_cfg, seed=seed)
            was_cached = run_cfg in _RUN_CACHE
            res = run_experiment(run_cfg)
            all_runs.append({"experiment": exp_name, "setting": label, "seed": seed,
                             **{k: res[k] for k in METRIC_KEYS}, "lbfgs_status": res["lbfgs_status"],
                             "lbfgs_evals": res["lbfgs_evals"]})
            note = "reused" if was_cached else f"{res['runtime_s']:.0f} s"
            print(f"{exp_name:>9s} | {label:<26s} | seed {seed} | alpha {res['alpha']:.5f} | "
                  f"rel. err {res['alpha_rel_err']:.2e} | {note}")
print(f"\nPart 2 training finished in {(time.perf_counter() - t_start) / 60:.1f} min")

## 15. Results table

Mean $\pm$ sample standard deviation over seeds. *CRLB* is the idealised lower bound on the relative standard
deviation of $\hat\alpha$ from section 13 (averaged over seeds; not defined for noise-free data). The table is also
saved to `results/` as raw CSV (one row per run), summary CSV, and a LaTeX `booktabs` table ready for `\input{}`.
Runtimes cover training only and depend on the hardware.

In [ ]:
def mean_std(values):
    v = np.asarray(values, dtype=float)
    return float(v.mean()), (float(v.std(ddof=1)) if v.size > 1 else 0.0)

def summarize(runs: list) -> list:
    groups = {}
    for r in runs:
        groups.setdefault((r["experiment"], r["setting"]), []).append(r)
    rows = []
    for (exp, setting), rs in groups.items():
        row = {"experiment": exp, "setting": setting, "n_seeds": len(rs)}
        for k in METRIC_KEYS:
            row[k + "_mean"], row[k + "_std"] = mean_std([r[k] for r in rs])
        row["lbfgs_status"] = "; ".join(sorted({r["lbfgs_status"] for r in rs}))
        rows.append(row)
    return rows

def fmt_pm(mean: float, std: float, latex: bool = False) -> str:
    # mean ± std with a shared power of ten, e.g. (3.12 ± 0.85)e-03
    ref = abs(mean) if mean != 0 else std
    if ref == 0:
        return "$0$" if latex else "0"
    e = int(math.floor(math.log10(ref)))
    m, s = mean / 10**e, std / 10**e
    if latex:
        return rf"$({m:.2f} \pm {s:.2f})\times 10^{{{e}}}$"
    return f"({m:.2f} ± {s:.2f})e{e:+03d}"

def table_cells(r: dict) -> list:
    crlb = f"{r['alpha_crlb_rel_mean']:.1e}" if r["alpha_crlb_rel_mean"] > 0 else "-"
    return [EXP_TITLES[r["experiment"]].split(" (")[0], r["setting"],
            f"{r['alpha_mean']:.4f} ± {r['alpha_std']:.4f}",
            fmt_pm(r["alpha_rel_err_mean"], r["alpha_rel_err_std"]), crlb,
            fmt_pm(r["field_rel_l2_mean"], r["field_rel_l2_std"]),
            fmt_pm(r["residual_mse_mean"], r["residual_mse_std"]),
            f"{r['runtime_s_mean']:.1f} ± {r['runtime_s_std']:.1f}"]

def print_table(rows: list) -> None:
    header = ["experiment", "setting", "alpha_hat", "alpha rel. err", "CRLB", "field rel. L2", "residual MSE", "runtime [s]"]
    body = [table_cells(r) for r in rows]
    widths = [max(len(h), *(len(line[i]) for line in body)) for i, h in enumerate(header)]
    sep = "-+-".join("-" * w for w in widths)
    print(" | ".join(h.ljust(w) for h, w in zip(header, widths)))
    print(sep)
    for i, line in enumerate(body):
        if i > 0 and line[0] != body[i - 1][0]:
            print(sep)
        print(" | ".join(c.ljust(w) for c, w in zip(line, widths)))

def latex_label(text: str) -> str:
    text = text.replace("%", r"\%").replace("_", r"\_")          # escape LaTeX special characters first
    return text.replace(r"N\_obs", r"$N_{\mathrm{obs}}$")

def write_latex_table(rows: list, path: str) -> None:
    n = rows[0]["n_seeds"]
    lines = [r"\begin{table}[htbp]", r"\centering", r"\small",
             rf"\caption{{Inverse PINN for the 1D heat equation: mean $\pm$ standard deviation over {n} seeds.}}",
             r"\label{tab:inverse-pinn-heat}",
             r"\begin{tabular}{llcccc}", r"\toprule",
             r"Experiment & Setting & $|\hat\alpha-\alpha|/\alpha$ & Rel.\ $L^2$ error & Residual MSE & Runtime [s] \\",
             r"\midrule"]
    previous = None
    for r in rows:
        if previous is not None and r["experiment"] != previous:
            lines.append(r"\midrule")
        name = EXP_TITLES[r["experiment"]].split(" (")[0] if r["experiment"] != previous else ""
        lines.append(" & ".join([name, latex_label(r["setting"]),
                                 fmt_pm(r["alpha_rel_err_mean"], r["alpha_rel_err_std"], latex=True),
                                 fmt_pm(r["field_rel_l2_mean"], r["field_rel_l2_std"], latex=True),
                                 fmt_pm(r["residual_mse_mean"], r["residual_mse_std"], latex=True),
                                 rf"${r['runtime_s_mean']:.1f} \pm {r['runtime_s_std']:.1f}$"]) + r" \\")
        previous = r["experiment"]
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

summary_rows = summarize(all_runs)
print_table(summary_rows)
lbfgs_runs = [r for r in all_runs if r["lbfgs_status"] != "not used"]
if lbfgs_runs:
    print("\nL-BFGS per seed: " + "; ".join(f"seed {r['seed']}: {r['lbfgs_status']}, {r['lbfgs_evals']} function evaluations"
                                         for r in lbfgs_runs))

OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)
with open(os.path.join(OUT_DIR, "runs_raw.csv"), "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(all_runs[0].keys()))
    writer.writeheader(); writer.writerows(all_runs)
with open(os.path.join(OUT_DIR, "summary.csv"), "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader(); writer.writerows(summary_rows)
write_latex_table(summary_rows, os.path.join(OUT_DIR, "results_table.tex"))
print(f"\nSaved: {', '.join(sorted(os.listdir(OUT_DIR)))} in ./{OUT_DIR}/ (Colab: folder icon on the left to download)")

## 16. Result plots

**Figure A.** One column per experiment, one row per metric (log scale). Grey dots are individual seeds, blue markers
mean $\pm$ std, red bars the CRLB for $\alpha$.
**Figure B.** Relative error of $\alpha$ during training (seed 0) for every setting: how fast, and how far, $\alpha$ converges.

In [ ]:
def plot_experiment_summary(all_runs: list, rows: list) -> None:
    exps = list(EXPERIMENTS)
    metrics = [("alpha_rel_err", r"$|\hat\alpha-\alpha|/\alpha$"), ("field_rel_l2", r"field rel. $L^2$ error"),
               ("residual_mse", "residual MSE")]
    fig, axes = plt.subplots(len(metrics), len(exps), figsize=(4.4 * len(exps), 3.0 * len(metrics)), squeeze=False)
    for j, exp in enumerate(exps):
        exp_rows = [r for r in rows if r["experiment"] == exp]
        for i, (key, ylabel) in enumerate(metrics):
            ax = axes[i, j]
            for k, r in enumerate(exp_rows):
                vals = [run[key] for run in all_runs if run["experiment"] == exp and run["setting"] == r["setting"]]
                jitter = np.linspace(-0.12, 0.12, len(vals)) if len(vals) > 1 else np.zeros(1)
                ax.scatter(k + jitter, vals, s=14, color="0.6", zorder=2, label="seeds" if k == 0 else None)
                ax.errorbar(k, r[key + "_mean"], yerr=r[key + "_std"], fmt="o", color="tab:blue", ms=6, capsize=5,
                            zorder=3, label="mean ± std" if k == 0 else None)
                if key == "alpha_rel_err" and r["alpha_crlb_rel_mean"] > 0:
                    ax.plot([k - 0.28, k + 0.28], [r["alpha_crlb_rel_mean"]] * 2, color="tab:red", lw=2, zorder=1,
                            label="CRLB" if not any(h.get_label() == "CRLB" for h in ax.lines) else None)
            ax.set_yscale("log")
            ax.set_xticks(range(len(exp_rows)))
            ax.set_xticklabels([r["setting"] for r in exp_rows] if i == len(metrics) - 1 else [],
                               rotation=25, ha="right", fontsize=8)
            ax.set_xlim(-0.5, len(exp_rows) - 0.5)
            if j == 0:
                ax.set_ylabel(ylabel)
            if i == 0:
                ax.set_title(EXP_TITLES[exp], fontsize=10)
    axes[0, 0].legend(fontsize=7, loc="best")
    fig.suptitle(f"Figure A - one-factor-at-a-time experiments ({len(SEEDS)} seeds each)", y=1.0)
    plt.tight_layout()
    plt.show()

def plot_alpha_trajectories(seed: int = 0) -> None:
    fig, axes = plt.subplots(1, len(EXPERIMENTS), figsize=(4.4 * len(EXPERIMENTS), 3.4), sharey=True)
    for ax, (exp, settings) in zip(axes, EXPERIMENTS.items()):
        for label, c in settings:
            res = _RUN_CACHE[replace(c, seed=seed)]
            h = res["history"]
            err = np.abs(np.asarray(h["alpha"]) - c.alpha_true) / c.alpha_true
            ax.semilogy(h["iter"], np.maximum(err, 1e-7), lw=1.2, label=label)
            if "lbfgs" in h["phase"]:
                ax.axvline(h["iter"][h["phase"].index("lbfgs")], color="0.5", ls=":", lw=1)
        ax.set(title=EXP_TITLES[exp], xlabel="iteration (L-BFGS: function evaluations)")
        ax.title.set_fontsize(10)
        ax.legend(fontsize=7)
    axes[0].set_ylabel(r"$|\alpha-\alpha_{true}|/\alpha_{true}$")
    fig.suptitle(f"Figure B - convergence of alpha (seed {seed})", y=1.02)
    plt.tight_layout()
    plt.show()

plot_experiment_summary(all_runs, summary_rows)
plot_alpha_trajectories(seed=SEEDS[0])

## 17. Diagnostics: identifiability and loss imbalance

**Where is the information about $\alpha$?** The sensitivity $|\partial u/\partial\alpha|=\pi^2 t\,e^{-\alpha\pi^2t}\sin(\pi x)$
is zero at $t=0$ (the initial state does not depend on $\alpha$), zero at the two ends of the rod, and small at late times
(the signal has decayed). It peaks at $x=\tfrac12$, $t^\ast=1/(\alpha\pi^2)\approx0.20$. Measurements near that region are
the most informative. The right panel shows the CRLB as a function of the number of measurements (averaged over 20 random
measurement layouts; no training involved), with the PINN errors from Part 2 overlaid.

**Loss imbalance.** For each loss term we measure $\|\nabla_\theta(w_k\mathcal L_k)\|$ — how hard that term pulls on the network
weights — and $|\partial(w_k\mathcal L_k)/\partial\log\alpha|$, before and after training. Very different magnitudes mean one
term dominates the updates (Wang et al., 2021).

In [ ]:
def crlb_curve(n_values, noise_level, n_layouts=20):
    means = []
    for n in n_values:
        vals = []
        for s in range(n_layouts):
            c = replace(BASE, n_obs=n, noise_level=noise_level, seed=100 + s)
            vals.append(alpha_crlb_rel({"obs": generate_observations(c)}, c))
        means.append(np.mean(vals))
    return np.asarray(means)

def plot_identifiability() -> None:
    X, T = eval_grid(BASE.n_eval)
    S = (PI**2 * T * reference_solution(X, T, BASE.alpha_true)).abs()
    t_star = 1.0 / (BASE.alpha_true * PI**2)
    obs = generate_observations(BASE)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
    ax = axes[0]
    im = ax.pcolormesh(to_np(X), to_np(T), to_np(S), shading="auto", cmap="magma")
    ax.scatter(to_np(obs["x"]), to_np(obs["t"]), s=18, c="cyan", edgecolors="k", linewidths=0.5, label="measurements (seed 0)")
    ax.axhline(t_star, color="w", ls="--", lw=1)
    ax.text(0.02, t_star + 0.02, r"$t^*=1/(\alpha\pi^2)$", color="w", fontsize=9)
    fig.colorbar(im, ax=ax)
    ax.set(xlabel="x", ylabel="t", title=r"Sensitivity $|\partial u/\partial\alpha|$")
    ax.legend(loc="upper right", fontsize=8)

    ax = axes[1]
    n_grid = np.array([10, 20, 50, 100, 200, 500])
    for p, color in ((0.01, "tab:blue"), (0.05, "tab:orange")):
        ax.loglog(n_grid, crlb_curve(n_grid, p), color=color, lw=1.5, label=f"CRLB, {p:.0%} noise")
    for r in all_runs:                      # PINN results from Part 2 at the matching settings
        c = replace(BASE, seed=r["seed"])
        if r["experiment"] == "n_obs":
            ax.scatter(int(r["setting"].split("=")[1]), r["alpha_rel_err"], color="tab:blue", s=18, marker="o")
        if r["experiment"] == "noise" and r["setting"] == "5% noise":
            ax.scatter(BASE.n_obs * 1.08, r["alpha_rel_err"], color="tab:orange", s=18, marker="s")
    ax.scatter([], [], color="tab:blue", marker="o", s=18, label="PINN, 1% noise (Part 2)")
    ax.scatter([], [], color="tab:orange", marker="s", s=18, label="PINN, 5% noise (Part 2)")
    ax.set(xlabel="number of measurements", ylabel=r"relative error / relative sd of $\hat\alpha$",
           title="PINN error vs. statistical lower bound")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

def loss_gradient_norms(model: InversePINN, data: dict, cfg: Config) -> dict:
    params = list(model.u_net.parameters())
    with reference_locked():
        _, parts = compute_losses(model, data, cfg)
    w = loss_weights(cfg)
    out = {}
    for k in LOSS_NAMES:
        term = w[k] * parts[k]
        g_net = torch.autograd.grad(term, params, retain_graph=True, allow_unused=True)
        g_alpha = torch.autograd.grad(term, model.log_alpha, retain_graph=True, allow_unused=True)[0]
        net_norm = torch.sqrt(sum((g ** 2).sum() for g in g_net if g is not None)).item()
        out[k] = (net_norm, 0.0 if g_alpha is None else g_alpha.abs().item())
    return out

plot_identifiability()

# Loss imbalance for the baseline: untrained network vs the model trained in Part 1
fresh = build_model(cfg)
g0, g1 = loss_gradient_norms(fresh, data, cfg), loss_gradient_norms(model, data, cfg)
print("term  | ||grad_theta||  init -> trained | |d/d log(alpha)|  init -> trained")
for k in LOSS_NAMES:
    print(f"{k:<5s} | {g0[k][0]:10.2e} -> {g1[k][0]:9.2e}       | {g0[k][1]:10.2e} -> {g1[k][1]:9.2e}")
print("\nOnly the PDE term produces a gradient for alpha: the data inform alpha only through the network.")

## 18. Discussion

> The numbers below come from a **reference run of this notebook on a CPU** (PyTorch 2.14, `float32`, seeds 0–2,
> 5000 Adam iterations). A GPU run will give slightly different values, because floating-point results differ between
> hardware. The qualitative conclusions should hold; compare them with your own table from section 15.

**Reference results** (mean $\pm$ sample std over 3 seeds; CRLB for the baseline layout is $2.0\times10^{-3}$)

| Setting | $\alpha$ rel. error | Field rel. $L^2$ | Residual MSE |
|---|---|---|---|
| Baseline: 1 % noise, 50 obs., weights $(1,1,1,1)$, Adam | $(3.6\pm1.7)\times10^{-3}$ | $(4.2\pm1.8)\times10^{-3}$ | $(3.2\pm1.6)\times10^{-5}$ |
| 0 % noise | $(4.1\pm2.3)\times10^{-3}$ | $(4.1\pm2.0)\times10^{-3}$ | $(3.1\pm1.6)\times10^{-5}$ |
| 5 % noise (CRLB $9.9\times10^{-3}$) | $(3.2\pm1.9)\times10^{-3}$ | $(7.4\pm2.2)\times10^{-3}$ | $(3.0\pm1.5)\times10^{-5}$ |
| 20 observations | $(1.4\pm1.7)\times10^{-2}$ | $(1.1\pm1.2)\times10^{-2}$ | $(3.3\pm1.7)\times10^{-5}$ |
| 100 observations | $(2.8\pm1.5)\times10^{-3}$ | $(4.1\pm2.3)\times10^{-3}$ | $(2.7\pm1.2)\times10^{-5}$ |
| Weights $(1,1,1,10)$, data-heavy | $(1.1\pm1.3)\times10^{-3}$ | $(4.4\pm0.4)\times10^{-3}$ | $(5.0\pm0.9)\times10^{-5}$ |
| Weights $(10,1,1,1)$, physics-heavy | $(1.0\pm0.3)\times10^{-1}$ | $(6.2\pm1.5)\times10^{-2}$ | $(3.0\pm0.5)\times10^{-5}$ |
| Weights $(0.1,1,1,1)$, physics-light | $(1.8\pm0.9)\times10^{-3}$ | $(2.8\pm0.5)\times10^{-3}$ | $(1.3\pm0.6)\times10^{-4}$ |
| Adam + L-BFGS | $(2.1\pm2.2)\times10^{-3}$ | $(2.9\pm1.9)\times10^{-3}$ | $(1.9\pm2.0)\times10^{-5}$ |

Each Adam run took about 56 s on one CPU thread; L-BFGS added 1–26 s.

### 18.1 Identifiability

* **Where the information is.** $\alpha$ only controls the decay rate $e^{-\alpha\pi^2t}$. The sensitivity
  $\partial u/\partial\alpha=-\pi^2t\,u$ is zero at $t=0$ and at $x\in\{0,1\}$, and small at late times, where the signal
  has decayed. Measurements taken only at $t=0$ or only on the boundary would leave $\alpha$ completely undetermined
  (the loss would be flat in $\alpha$). Measurements taken only at late times would determine it poorly. The most informative
  region is around $x=\tfrac12$, $t^\ast=1/(\alpha\pi^2)\approx0.2$ (section 17).
* **How the information reaches $\alpha$.** $\alpha$ receives gradient only from the PDE term; the gradient-norm table in
  section 17 shows exactly zero for the other terms. The chain is *data → network → residual → $\alpha$*: if the network does not
  reproduce the decay seen in the data, $\alpha$ cannot be right, whatever the data contain.
* **Statistical versus practical identifiability.** The CRLB answers "how well could $\alpha$ be known from these data at
  best?". A Monte Carlo check of 300 least-squares fits of the exact solution family matched the bound closely:
  relative standard deviation $1.62\times10^{-3}$ versus a bound of $1.65\times10^{-3}$ at 1 % noise, and $8.1\times10^{-3}$ versus
  $8.2\times10^{-3}$ at 5 %. The PINN does not always reach this. In the reference run, the 20-point layout of seed 2 gave
  $\hat\alpha=0.483$ (3.4 % error), although a least-squares fit of the exact family to the *same* 20 noisy points gives
  0.5012 (0.2 %). That layout had only 2 points in the high-sensitivity region
  ($0.05<t<0.5$, $0.2<x<0.8$). The information was present, but a PINN that must also learn the whole field from sparse
  data did not extract it. For a PINN, practical identifiability is stricter than statistical identifiability.

### 18.2 Sensitivity to noise and data coverage

* **Noise.** Going from 0 % to 5 % noise hardly changed the $\alpha$ error: $4.1$, $3.6$ and $3.2\times10^{-3}$ on average.
  The error at 0 % noise is not zero, so at these noise levels **optimisation and approximation error dominate**, not
  measurement noise. At 1 % noise the PINN was roughly $2\times$ above the CRLB. At 5 % its errors were below the CRLB
  standard deviation. With 3 seeds that is not a contradiction: the bound constrains the spread over many noise
  realisations, not individual runs. Noise shows up more clearly in the reconstructed field: the relative $L^2$ error was
  $7.4\times10^{-3}$ at 5 % versus $4.2\times10^{-3}$ at 1 %, because the network partly follows the noise.
* **Coverage.** More measurements helped modestly on average: $1.4\times10^{-2}$, $3.6\times10^{-3}$ and $2.8\times10^{-3}$ for 20, 50 and 100
  points. The 20-point mean, however, is driven by the single poorly placed layout above. *Where* the measurements are
  matters at least as much as *how many* there are. Placing sensors around $t^\ast$ in the middle of the rod is the
  cheapest improvement.

### 18.3 Loss imbalance

* **The ratio $w_{\text{data}}/w_{\text{pde}}$ controls the $\alpha$ accuracy.** Ratio 0.1 gave an error of $1.0\times10^{-1}$, ratio 1
  gave $3.6\times10^{-3}$, and ratio 10 gave $1.8\times10^{-3}$ (physics-light) and $1.1\times10^{-3}$ (data-heavy).
* **Physics-heavy weighting is slow, not wrong.** With $w_{\text{pde}}=10$, $\alpha$ was still rising steadily at the end.
  For seed 0 it went from 0.26 at iteration 1000 to 0.44 at 5000. The cheapest way to lower a physics-dominated loss is to
  make the network satisfy the PDE for the *current, wrong* $\alpha$. The data and initial-condition losses then stay 30–100
  times higher than in the baseline, and $\alpha$ is dragged toward the truth only slowly. Under a fixed iteration budget this
  looks like failure.
* **The price of down-weighting physics.** The PDE residual grows ($1.3\times10^{-4}$ for physics-light versus $3.2\times10^{-5}$), so
  the reconstructed field obeys the heat equation less well. With noisier data, a large data weight would also fit more
  noise. No single weighting is best for every goal. Adaptive schemes that balance gradient magnitudes during training
  (Wang, Teng & Perdikaris, 2021) address this; section 17 shows the gradient magnitudes they work with.

### 18.4 Optimizers

L-BFGS never produced NaN and never needed the revert, so it counted as "stable". Its benefit, however, was inconsistent
in `float32`. For seeds 0 and 2 it stopped after only 18 and 10 function evaluations and barely changed the Adam solution.
For seed 1 it ran 688 evaluations, lowered $\mathcal L_{\text{pde}}$ from about $2\times10^{-5}$ to $7.7\times10^{-7}$, and cut the $\alpha$ error
from $4.6\times10^{-3}$ to $1.8\times10^{-4}$. The early stops are most likely a precision effect: the line search can no longer find a
decrease that `float32` can resolve. This explanation was not verified here. Setting `DTYPE = torch.float64` is the usual
remedy. The cost was modest: 1–26 s on top of about 56 s of Adam.

### 18.5 Limitations

* A single scalar parameter and a smooth analytic solution. Realistic inverse problems — a spatially varying $\alpha(x)$,
  unknown initial data, 2D/3D domains — are much harder.
* The synthetic data come from the same PDE the network enforces, so there is no model error. The noise is i.i.d.
  Gaussian and scaled to the data's standard deviation.
* Three seeds give only rough standard deviations. Differences below about $10^{-3}$ in $\alpha$ error should not be
  interpreted.
* One factor is varied at a time, so interactions are not studied (e.g. data-heavy weighting at 5 % noise).
* The iteration budget is fixed at 5000. Some settings, notably physics-heavy, are not converged. The table compares
  accuracy *after a fixed budget*, not accuracy at convergence.
* Boundary and initial conditions are soft constraints, the collocation points are fixed, and the precision is `float32`.
  There is no uncertainty estimate for $\hat\alpha$ beyond the seed-to-seed spread.
* Runtimes depend on hardware and are only comparable within one session.

### 18.6 Possible extensions

`float64` with L-BFGS; a hard-constraint ansatz $u_\theta=\sin(\pi x)+t\,x(1-x)\,N_\theta(x,t)$, which removes
$\mathcal L_{\text{bc}}$ and $\mathcal L_{\text{ic}}$; adaptive loss weights; measurement design around $t^\ast$; deep ensembles or Bayesian PINNs
for uncertainty in $\hat\alpha$; and a spatially varying diffusivity $\alpha(x)$ represented by a second network.

### References

* M. Raissi, P. Perdikaris, G. E. Karniadakis (2019). Physics-informed neural networks. *J. Comput. Phys.* 378, 686–707.
* G. E. Karniadakis et al. (2021). Physics-informed machine learning. *Nature Reviews Physics* 3, 422–440.
* S. Wang, Y. Teng, P. Perdikaris (2021). Understanding and mitigating gradient flow pathologies in physics-informed
  neural networks. *SIAM J. Sci. Comput.* 43(5), A3055–A3081.
* J. Nocedal, S. J. Wright (2006). *Numerical Optimization*, 2nd ed. Springer (L-BFGS, Wolfe line search).
* S. M. Kay (1993). *Fundamentals of Statistical Signal Processing: Estimation Theory*. Prentice Hall (Cramér–Rao bound).